# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [8]:
# imports
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import ollama


In [9]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OLLAMA_API = "http://localhost:11434/api/chat"
HEADERS = {"Content-Type": "application/json"}
MODEL = "llama3.2"

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()

API key looks good so far


In [11]:
# here is the question; type over this to ask something new

system_prompt = "Your are given a random question\n"

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

def stream_answer(model, system_prompt, question):
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
      ]

    api = openai
    
    if model is MODEL_LLAMA:
        api = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    stream = api.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)


In [7]:
# Get gpt-4o-mini to answer, with streaming
stream_answer(MODEL_GPT, system_prompt, question)

This code snippet is using Python's generator functionality combined with a set comprehension to yield values from a set of authors extracted from a list (or similar iterable) of books. Let's break it down step by step:

1. **Set Comprehension**: 
   - The expression `{book.get("author") for book in books if book.get("author")}` is a set comprehension. It creates a set of unique authors from the `books` collection.
   - It iterates through each `book` in `books`.
   - For each `book`, it tries to get the value associated with the key `"author"` using `book.get("author")`.
   - The `if book.get("author")` condition filters out books where the author is `None` or an empty string, ensuring that only books with valid authors are included.

2. **Yield from**: 
   - The `yield from` statement is a way to yield all values from an iterable. In this case, it will yield each author from the set created by the set comprehension.
   - Using `yield from` means that this function (which contains this code) is a generator function. Thus, it allows you to iterate over the results one at a time, which is memory efficient especially when dealing with large data sets.

### Why Use This Code?

- **Efficiency**: By using a set comprehension, it automatically handles duplicate authors, which means you get a collection of unique authors without needing to filter duplicates manually.
- **Filter Valid Authors**: The condition in the comprehension ensures that only books with valid authors are included in the final result.
- **Lazy Evaluation**: The `yield from` allows for lazy evaluation, meaning that the authors are generated and returned one at a time, rather than creating a full list in memory all at once.

### Example

Assuming `books` is a list of dictionaries structured like:
python
books = [
    {"title": "Book One", "author": "Author A"},
    {"title": "Book Two", "author": "Author B"},
    {"title": "Book Three", "author": "Author A"},
    {"title": "Book Four", "author": None},
    {"title": "Book Five", "author": "Author C"},
]


The code:
python
yield from {book.get("author") for book in books if book.get("author")}

Would effectively yield:
- "Author A"
- "Author B"
- "Author C"

Resulting in a generator of unique author names, excluding any books that do not have an author or have 'None' as an author.

In [12]:
# Get Llama 3.2 to answer
stream_answer(MODEL_LLAMA, system_prompt, question)

**Code Explanation**

This is a Python expression that uses several advanced features to extract specific data from a collection of books. Let's break it down:

* `yield from`: This keyword is used to delegate the execution of a generator or iterator to another iterable (in this case, a dictionary comprehension).
* `{book.get("author") for book in books if book.get("author")}`: This is an expression that generates a new dictionary containing only the values associated with the key `"author"`. Here's what's happening:
	1. `for book in books`: This loop iterates over each dictionary (`book`) in the collection of books.
	2. `if book.get("author")`: For each dictionary, this line uses the `.get()` method to attempt to retrieve the value associated with the key `"author"`. If it finds a match, the expression includes that author's name in the subsequent dictionary.
3. `yield from ...`: This keyword transfers control of iteration over to the delegated generator (in this case, the one generated by the dictionary comprehension). The `yield from` statement sends values from each delegation to the current context.

**In Summary**

This code uses a combination of generators and dictionary comprehensions to extract author names from a collection of books. It iterates over each book in the collection, extracts the `"author"` value for each one that exists, and yields all these author names together.

However, there's an important subtlety: `{book.get("author") for book in books if book.get("author")}` will always yield a dictionary with arbitrary behavior in most cases. It will indeed contain some values (some of which are strings) but in general this is used when we don't need a full object and we can deal well using key value pairs.

The main goal here is likely to be filtering the collection on `book.get("author")` being truthy, while then still getting these values:

python
filtered_authors = yield from {book.get("author") for book in books if book.get("author")}


Is typically used when we can get away without keeping the whole `book` dictionary to start with.

Here's a refactored version of this function using list comprehension where it makes sense:

python
filtered_authors = [book["author"] for book in books if "author" in book]
for author in filtered_authors:
    yield author

And lastly the original one is much more suitable when you really need both the entire data and key-value pair.